<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/BallTrap_Global_D_Joint_Direction_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global-D Ball Trap: Standalone Training Notebook

This notebook contains the **brain implementation, continuous-relaxation trainer, data preparation, train/validation/test split, evaluation, and compilation**. It does not import any project Python files. Replace the example pairs below with your own data, or read a CSV with `input` and `output` columns.

The default interface is text in/text out at the character level. Each example is `(input_text, desired_output_text)`. The continuous trainer learns global hash directions and start positions, profiles the shared bucket table, then rounds the parameters and compiles the table into native ball-trap deflectors.

Run cells in order. `D` is one shared global node space. `hash_radius` controls how far each input character looks for context. `direction_range` bounds the learned context coefficients. `cap` is raised automatically when targets need more emission steps per input ball.


## 1. Dependencies

The trainer source is included in this notebook. The only non-standard Python package it needs is SciPy (NumPy is generally already present in notebook environments). This cell installs a missing dependency in the notebook runtime; it does not download project files.


In [ ]:
import importlib.util

if importlib.util.find_spec("numpy") is None or importlib.util.find_spec("scipy") is None:
    get_ipython().run_line_magic("pip", "install numpy scipy")


## 2. Complete brain and trainer implementation

This cell is the complete global-D brain and continuous trainer implementation. It is intentionally in the notebook so you can train without importing `ball_trap.py` or any other project module.


In [ ]:
import copy
import csv
import difflib
import gzip
import json
import math
import random
import time

import numpy as np
from scipy.sparse import csr_matrix

from itertools import chain
from numbers import Integral



def encode_bpe(tokenizer, value):
    """Get integer IDs from text or an already encoded sequence."""
    tokenizer = getattr(tokenizer, "backend_tokenizer", tokenizer)
    if isinstance(value, str):
        if hasattr(tokenizer, "encode_ordinary"):
            value = tokenizer.encode_ordinary(value)
        else:
            try:
                value = tokenizer.encode(value, add_special_tokens=False)
            except TypeError as error:
                if "keyword" not in str(error):
                    raise
                value = tokenizer.encode(value)
    ids = tuple(getattr(value, "ids", value))
    if any(not isinstance(token, Integral) or isinstance(token, bool) or token < 0 for token in ids):
        raise ValueError("BPE inputs and outputs must be text or a flat sequence of token IDs.")
    return tuple(map(int, ids))


def TokenizeInput(input, alphabet):
    """Greedy non-overlapping tokenization: earliest start, then longest, then alphabet order."""
    input_str = str(input)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda m: (m[0], m[1], m[2]))
    out, last_end = [], 0
    for index, neg_len, _, item in matches:
        if index >= last_end:
            out.append(item)
            last_end = index - neg_len
    return out

class BallTrapBrain:
    def __init__(
        self, alphabet, output, D=1024, max_input_len=6,
        hash_radius=3, bounce=10, cap=10, direction_range=16, *, tokenizer=None
    ):
        self.alphabet = list(alphabet)
        self.output = list(output)
        self.tokenizer = getattr(tokenizer, "backend_tokenizer", tokenizer)
        self._character_alphabet = all(isinstance(t, str) and len(t) == 1 for t in self.alphabet)
        if self.tokenizer is not None:
            if any(not isinstance(t, Integral) or isinstance(t, bool) or t < 0
                   for t in chain(self.alphabet, self.output)):
                raise ValueError("With tokenizer=, alphabet and output must contain integer token IDs.")
            self.alphabet = list(map(int, self.alphabet))
            self.output = list(map(int, self.output))
        self.index = {t: i for i, t in enumerate(self.alphabet)}

        self.D = int(D)
        self.max_input_len = int(max_input_len)
        self.hash_radius = int(hash_radius)
        if self.D < 1:
            raise ValueError("D must be a positive global node count.")
        self.cap = int(cap)
        self.direction_range = int(direction_range)

        self.ball_energy = bounce * 2

        # Shared relative-position hash features.
        # Center offset is unused because s[token_i] is the center feature.
        self.hash_directions = np.random.randint(
            -self.direction_range,
            self.direction_range + 1,
            size=(2 * self.hash_radius + 1, len(self.alphabet)),
            dtype=np.int64,
        )
        self.hash_directions[self.hash_radius] = 0
        self.hash_start_positions = np.random.randint(
            0, self.D, size=len(self.alphabet), dtype=np.int64
        )

        # deflector = [destination, value]; value is an int (bounce energy) or str (hole symbol)
        self.deflectors_list = [
            [random.randint(0, self.D - 1), bounce] for _ in range(self.D)
        ]

    @classmethod
    def from_tokenizer(cls, tokenizer, **kwargs):
        """Build a brain whose input and output alphabets are tokenizer IDs."""
        tokenizer = getattr(tokenizer, "backend_tokenizer", tokenizer)
        if hasattr(tokenizer, "get_vocab"):
            vocab = sorted(set(tokenizer.get_vocab().values()))
        elif hasattr(tokenizer, "n_vocab"):
            vocab = range(tokenizer.n_vocab)
        else:
            raise ValueError("Pass the tokenizer's integer vocabularies to BallTrapBrain(..., tokenizer=...).")
        return cls(vocab, vocab, tokenizer=tokenizer, **kwargs)

    @property
    def empty_fragment(self):
        return "" if self.tokenizer is None else ()

    def input_indices(self, value):
        if self.tokenizer is not None:
            ids = encode_bpe(self.tokenizer, value)
            try:
                return [self.index[token] for token in ids]
            except KeyError as error:
                raise ValueError(f"BPE token ID {error.args[0]} is outside the brain's input alphabet.") from None
        tokens = str(value) if self._character_alphabet else TokenizeInput(value, self.alphabet)
        return [self.index[token] for token in tokens if token in self.index]

    def output_symbols(self, value):
        if self.tokenizer is not None:
            return encode_bpe(self.tokenizer, value)
        if not isinstance(value, str):
            raise ValueError("Provide text targets or configure tokenizer= to use BPE token IDs.")
        return value

    def join_fragments(self, fragments):
        if self.tokenizer is None:
            return "".join(fragments)
        ids = list(chain.from_iterable(fragments))
        if not ids:
            return ""
        try:
            return self.tokenizer.decode(ids, skip_special_tokens=False)
        except TypeError as error:
            if "keyword" not in str(error):
                raise
            return self.tokenizer.decode(ids)

    def start_nodes(self, input):
        ids = self.input_indices(input)
        if self.tokenizer is not None and len(ids) > self.max_input_len:
            raise ValueError(f"Input has {len(ids)} BPE tokens; max_input_len is {self.max_input_len}.")
        nodes = []

        for i, tok_i in enumerate(ids):
            local = int(self.hash_start_positions[tok_i])
            for delta in range(-self.hash_radius, self.hash_radius + 1):
                if delta == 0:
                    continue
                j = i + delta
                if 0 <= j < len(ids):
                    local += int(self.hash_directions[delta + self.hash_radius, ids[j]])
            # All positions route into the same global D-node system.
            nodes.append(local % self.D)

        return nodes

    def walk_symbols(self, node):
        """Return a native fragment before any BPE decoding."""
        out, dl, be, steps = [], self.deflectors_list, self.ball_energy, 0
        while be > 0 and steps < self.cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                out.append(val)
                be += 1 - dist
            node, steps = dest, steps + 1
        if self.tokenizer is not None:
            return tuple(map(int, out))
        return "".join(out)

    def walk(self, node):
        fragment = self.walk_symbols(node)
        return fragment if self.tokenizer is None else self.join_fragments([fragment])

    def compute_input(self, input):
        nodes = self.start_nodes(input)
        if self.tokenizer is None:
            return "".join(self.walk(node) for node in nodes)
        return self.join_fragments(self.walk_symbols(node) for node in nodes)

    def build_table(self):
        """T[k]: the string emitted from each bucket."""
        return [self.walk(k) for k in range(self.D)]

    def display_deflectors(self, n=20):
        for i, defl in enumerate(self.deflectors_list[:n]):
            print(f"Deflector {i}: {defl}")


def evaluate(brain, cases):
    exact = ratio = 0.0
    for inp, ans in cases:
        pred = brain.compute_input(inp)
        exact += pred == ans
        ratio += difflib.SequenceMatcher(None, pred, ans).ratio()
    return exact / len(cases), ratio / len(cases)




DEFAULT_LEARNING_RATE = 0.22


def fragment_array(fragments):
    """Keep a tuple of BPE IDs as one object even when fragments are equally long."""
    values = list(fragments)
    result = np.empty(len(values), dtype=object)
    result[:] = values
    return result


class CaseMatrix:
    """One sparse row per ball. The center feature uses a start-position column."""

    def __init__(self, brain, cases, pieces=None, *, encode_targets=True, auto_cap=False):
        self.brain = brain
        self.cases = list(cases)
        if not self.cases:
            raise ValueError("Provide at least one example.")
        if pieces is not None and len(pieces) != len(self.cases):
            raise ValueError("Provide one segmentation per example.")
        self.targets = [
            target if isinstance(target, str) else brain.join_fragments([brain.output_symbols(target)])
            for _, target in self.cases
        ]
        tokens = [brain.input_indices(prompt) for prompt, _ in self.cases]
        self.lengths = np.array([len(row) for row in tokens], dtype=np.int64)
        too_long = np.flatnonzero(self.lengths > brain.max_input_len)
        if len(too_long):
            j = int(too_long[0])
            raise ValueError(f"Example {j + 1}: the prompt produces {self.lengths[j]} balls, "
                             f"but max_input_len is {brain.max_input_len}.")
        labels = []
        if encode_targets:
            labels, cap = self._encode_fragments(brain, pieces, auto_cap)
            if cap > brain.cap:
                self.brain = copy.copy(brain)
                self.brain.cap = cap
        padded = np.full((len(tokens), brain.max_input_len), -1, dtype=np.int64)
        for j, row in enumerate(tokens):
            padded[j, :len(row)] = row
        example, position = np.nonzero(padded >= 0)
        self.example = example
        self.position = position
        self.labels = fragment_array(p for row in labels for p in row)
        self.starts = np.concatenate(([0], self.lengths.cumsum()))
        self.padded = padded

    def _encode_fragments(self, brain, pieces, auto_cap):
        cap = brain.cap
        targets = [brain.output_symbols(target) for _, target in self.cases]
        supplied = None if pieces is None else [
            [brain.output_symbols(fragment) for fragment in row] for row in pieces
        ]
        output_set = set(brain.output)
        unit = "BPE tokens" if brain.tokenizer is not None else "characters"
        for j, (length, target) in enumerate(zip(self.lengths, targets), start=1):
            n_balls = int(length)
            target_size = len(target)
            if n_balls == 0 and target_size:
                raise ValueError(f"Example {j}: the prompt produces no balls after tokenization, "
                                 f"so it cannot emit the target of {target_size} {unit}. "
                                 "Check the input alphabet.")
            if any(symbol not in output_set for symbol in target):
                raise ValueError(f"Example {j}: the target contains symbols outside the output alphabet.")
            if supplied is None:
                required = (target_size + n_balls - 1) // n_balls if n_balls else 0
            else:
                fragments = supplied[j - 1]
                flattened = tuple(symbol for fragment in fragments for symbol in fragment) if (
                    brain.tokenizer is not None
                ) else "".join(fragments)
                if len(fragments) != n_balls or flattened != target:
                    raise ValueError(f"Example {j}: provide one fragment per ball "
                                     f"({n_balls} fragments) that together reproduce the target.")
                required = max(map(len, fragments), default=0)
            if required > cap and not auto_cap:
                detail = (f"At most {n_balls * cap} {unit} can be emitted at this cap."
                          if supplied is None else f"The longest assigned fragment takes {required} steps.")
                raise ValueError(f"Example {j}: {n_balls} balls after tokenization, "
                                 f"{target_size} target {unit}, cap={cap} steps per ball. "
                                 f"{detail} This example needs cap >= {required}.")
            cap = max(cap, required)
        if supplied is None:
            supplied = [self.split(target, length, cap)
                        for length, target in zip(self.lengths, targets)]
        return supplied, cap

    @staticmethod
    def split(target, length, cap):
        if len(target) > length * cap:
            raise ValueError("Target exceeds the emission cap of the input balls.")
        if len(target) <= length:
            return [target[:0]] * (length - len(target)) + [
                target[i:i + 1] for i in range(len(target))
            ]
        result, offset, extra = [], 0, len(target) - length
        for _ in range(length):
            take = min(extra, cap - 1)
            result.append(target[offset:offset + 1 + take])
            extra -= take
            offset += 1 + take
        return result

    @staticmethod
    def feature_rows(padded, example, position, brain):
        radius = brain.hash_radius
        alphabet_size = len(brain.alphabet)
        rows, cols = [], []
        for band, delta in enumerate(range(-radius, radius + 1)):
            shifted = position + delta
            inside = (shifted >= 0) & (shifted < padded.shape[1])
            row = np.flatnonzero(inside)
            token = padded[example[row], shifted[row]]
            present = token >= 0
            rows.append(row[present])
            cols.append(band * alphabet_size + token[present])
        row = np.concatenate(rows)
        col = np.concatenate(cols)
        return csr_matrix(
            (np.ones(len(row)), (row, col)),
            shape=(len(position), (2 * radius + 1) * alphabet_size),
        )

    def strings(self, fragments):
        return [
            self.brain.join_fragments(fragments[a:b])
            for a, b in zip(self.starts[:-1], self.starts[1:])
        ]


class ContinuousBallTrainer:
    """Periodic spline relaxation followed by straight-through integer routing."""

    def __init__(self, brain, cases, *, pieces=None, recency_decay=None, auto_cap=False):
        if brain.D < 1 or brain.cap < 1 or not brain.alphabet:
            raise ValueError("Use at least one bucket, one step, and one input symbol.")
        self.train = CaseMatrix(brain, cases, pieces, auto_cap=auto_cap)
        self.brain = brain = self.train.brain
        self.width = brain.D
        self.radius = brain.hash_radius
        self.grid = self.width
        empty = brain.empty_fragment
        self.classes = fragment_array([empty] + sorted(set(self.train.labels) - {empty}))
        self.n_classes = len(self.classes)
        label_id = {fragment: i for i, fragment in enumerate(self.classes)}
        self.label = np.array([label_id[p] for p in self.train.labels], dtype=np.int64)
        # Every example, regardless of input position, competes in one global
        # bucket table. No position-indexed local-D blocks are created.
        self.features = CaseMatrix.feature_rows(
            self.train.padded, self.train.example, self.train.position, brain
        )
        self.y = self.label
        self.features_transpose = self.features.T.tocsr()
        self.table_size = self.width * self.n_classes

        initial = np.array(brain.hash_directions, dtype=np.float64, copy=True)
        initial[self.radius] = brain.hash_start_positions
        self.lower = np.full_like(initial, -brain.direction_range)
        self.upper = np.full_like(initial, brain.direction_range)
        self.lower[self.radius] = 0
        self.upper[self.radius] = self.width - 1
        scale = np.ones_like(initial)
        if recency_decay is not None:
            if recency_decay <= 0:
                raise ValueError("recency_decay must be positive.")
            scale *= np.exp(-np.abs(np.arange(2 * self.radius + 1) - self.radius)
                            / recency_decay)[:, None]
        self.lower = self.lower.ravel()
        self.upper = self.upper.ravel()
        self.initial = np.clip(initial.ravel(), self.lower, self.upper)
        self.scale = scale.ravel()
        self.shadow = self.initial.copy()
        self.weights = self.initial.astype(np.int64)
        self.table, _ = self.hard_table(self.weights)

    def quantize(self, shadow):
        return np.clip(np.rint(shadow), self.lower, self.upper).astype(np.int64)

    @staticmethod
    def spline(phases, width):
        """Four cubic B-spline weights and their derivatives on a circular grid."""
        center = np.floor(phases)
        f = phases - center
        g = 1 - f
        near = (center.astype(np.int64)[:, None] + np.arange(-1, 3)) % width
        weights = np.stack((g**3, 4 - 6*f*f + 3*f**3,
                            4 - 6*g*g + 3*g**3, f**3), axis=1) / 6
        derivatives = np.stack((-3*g*g, -12*f + 9*f*f,
                                12*g - 9*g*g, 3*f*f), axis=1) / 6
        return near, weights, derivatives

    def soft_profile(self, weights, batch_indices=None):
        if batch_indices is None:
            feat = self.features
            y = self.y
            feat_t = self.features_transpose
        else:
            feat = self.features[batch_indices]
            y = self.y[batch_indices]
            feat_t = feat.T.tocsr()

        phases = feat @ weights
        near, mass, derivatives = self.spline(phases, self.width)
        nodes = near
        keys = nodes * self.n_classes + y[:, None]
        counts = np.bincount(keys.ravel(), weights=mass.ravel(), minlength=self.table_size)
        counts = counts.reshape(-1, self.n_classes)
        totals = counts.sum(axis=1)
        # Tiny pseudocount makes the profile differentiable at empty class cells.
        epsilon = 1e-8
        smoothed_counts = counts + epsilon
        smoothed_totals = totals + self.n_classes * epsilon
        log_total = np.log(smoothed_totals)
        log_count = np.log(smoothed_counts)
        costs = log_total[nodes] - log_count.ravel()[keys]
        n = max(1, len(y))
        gradient = feat_t @ np.sum(derivatives * costs, axis=1) / n
        entropy = smoothed_totals @ log_total - smoothed_counts.ravel() @ log_count.ravel()
        return gradient, counts, float(entropy / n)

    def hard_table(self, weights):
        local = (self.features @ weights).astype(np.int64) % self.width
        keys = local * self.n_classes + self.y
        counts = np.bincount(keys, minlength=self.table_size).reshape(-1, self.n_classes)
        table = counts.argmax(axis=1)
        total = counts.sum(axis=1)
        nz = counts[counts > 0]
        occupancy_ce = (total[total > 0] @ np.log(total[total > 0])
                        - nz @ np.log(nz)) / max(1, len(self.y))
        return table, float(occupancy_ce)

    def hard_predict(self, cases, weights=None, table=None):
        weights = self.weights if weights is None else weights
        table = self.table if table is None else table
        features = CaseMatrix.feature_rows(
            cases.padded, cases.example, cases.position, cases.brain
        )
        local = (features @ weights).astype(np.int64) % self.width
        ids = table[local]
        return cases.strings(self.classes[ids])

    def soft_predict(self, cases, weights):
        """Read the profiled soft router by mixing bucket class distributions."""
        _, counts, _ = self.soft_profile(weights)
        sums = counts.sum(axis=1, keepdims=True)
        prob = np.divide(counts, sums, out=np.zeros_like(counts), where=sums > 0)
        prob[sums.ravel() == 0, 0] = 1
        features = CaseMatrix.feature_rows(
            cases.padded, cases.example, cases.position, cases.brain
        )
        near, mass, _ = self.spline(features @ weights, self.width)
        ids = (prob[near] * mass[..., None]).sum(axis=1).argmax(axis=1)
        return cases.strings(self.classes[ids])

    @staticmethod
    def accuracy(predictions, cases):
        return float(np.mean([pred == target for pred, target in zip(predictions, cases.targets)]))

    def fit(self, *, warmup=100, hard_steps=200, learning_rate=DEFAULT_LEARNING_RATE,
            integer_pull=0.004, validation=None, check_every=10, verbose=True, batch_size=26):
        if not math.isfinite(learning_rate) or learning_rate <= 0:
            raise ValueError("learning_rate must be a finite positive number.")
        if warmup < 0 or hard_steps < 0 or check_every < 1 or integer_pull < 0:
            raise ValueError("Use nonnegative step counts and integer pull, and positive cadence.")
        start = time.perf_counter()
        weights = self.shadow.copy()
        validation = self.train if validation is None else validation
        best = None
        self.history = []

        def checkpoint(step):
            nonlocal best
            discrete = self.quantize(weights)
            table, ce = self.hard_table(discrete)
            train_acc = self.accuracy(self.hard_predict(self.train, discrete, table), self.train)
            val_acc = self.accuracy(self.hard_predict(validation, discrete, table), validation)
            self.history.append(dict(step=step, train_exact=train_acc, validation_exact=val_acc, ce=ce))
            score = (val_acc, train_acc, -ce)
            if best is None or score > best[0]:
                best = (score, discrete, table, step)
            if verbose and (step in (0, warmup, warmup + hard_steps) or step % 100 == 0):
                print(f"step={step:4d}  hard train={train_acc:.3%}  "
                      f"hard validation={val_acc:.3%}  hard CE={ce:.4f}", flush=True)

        checkpoint(0)
        self.warm_shadow = weights.copy()
        n_samples = len(self.y)

        for steps, hard in ((warmup, False), (hard_steps, True)):
            first = np.zeros_like(weights)
            second = np.zeros_like(weights)
            for i in range(1, steps + 1):
                # Optional mini-batch selection
                if batch_size is not None and batch_size < n_samples:
                    batch_indices = np.random.choice(n_samples, batch_size, replace=False)
                else:
                    batch_indices = None

                discrete = self.quantize(weights)
                gradient, _, _ = self.soft_profile(discrete if hard else weights, batch_indices=batch_indices)
                # During hard training the spline derivative is the straight-through
                # surrogate. The checkpoint uses the integer forward values and hard table.
                gradient += (0.25 if hard else 1) * integer_pull * (weights - discrete)
                first = 0.9 * first + 0.1 * gradient
                second = 0.999 * second + 0.001 * gradient**2
                fraction = (i - 1) / max(1, steps - 1)
                rate = learning_rate * (0.2 + 0.8 * (1 + math.cos(math.pi * fraction)) / 2)
                if hard:
                    rate *= 0.65
                update = rate * (first / (1 - 0.9**i)) / (np.sqrt(second / (1 - 0.999**i)) + 1e-8)
                weights = np.clip(weights - self.scale * update, self.lower, self.upper)
                step = i + (warmup if hard else 0)
                if i == steps or step % check_every == 0:
                    checkpoint(step)
            if not hard:
                self.warm_shadow = weights.copy()
        self.shadow = weights
        _, self.weights, self.table, self.best_step = best
        self.seconds = time.perf_counter() - start
        return self

    def compile(self):
        """Produce native deflectors and verify every possible legal start node."""
        brain = copy.copy(self.brain)
        layout = self.weights.reshape(brain.hash_directions.shape)
        brain.hash_directions = layout.copy()
        brain.hash_start_positions = layout[self.radius].copy()
        brain.hash_directions[self.radius] = 0
        fragments = self.classes[self.table].ravel().tolist()
        sink = next((j for j, fragment in enumerate(fragments) if not fragment), self.grid)
        # A silent start cell can serve as the sink. Continuations live beyond the grid.
        deflectors = [[j, 0] for j in range(self.grid + int(sink == self.grid))]
        suffix_nodes = {}
        output_set = set(brain.output)
        required_energy = max(1, int(brain.ball_energy))
        for start, fragment in enumerate(fragments):
            if not fragment:
                continue
            if len(fragment) > brain.cap or any(symbol not in output_set for symbol in fragment):
                raise ValueError("A table fragment cannot be emitted by this brain.")
            path = [start]
            for i in range(1, len(fragment)):
                suffix = fragment[i:]
                if suffix not in suffix_nodes:
                    suffix_nodes[suffix] = len(deflectors)
                    deflectors.append([len(deflectors), 0])
                path.append(suffix_nodes[suffix])
            for node, dest, symbol in zip(path, path[1:] + [sink], fragment):
                deflectors[node] = [dest, str(symbol)]
            cost = 0
            for a, b in zip(path, path[1:]):
                cost += abs(b - a) - 1
                required_energy = max(required_energy, cost + 1)
        brain.ball_energy = required_energy
        brain.deflectors_list = [
            [j, -required_energy] if value == 0 else [dest, value]
            for j, (dest, value) in enumerate(deflectors)
        ]
        walk = brain.walk if brain.tokenizer is None else brain.walk_symbols
        for node, expected in enumerate(fragments):
            if walk(node) != expected:
                raise RuntimeError(f"Physical walk differs from bucket {node}.")
        return brain

## 3. Enter or load your examples

Edit `examples` directly, or set `CSV_PATH` to a CSV you already have. CSV input/output column names are configurable. The trainer itself remains self-contained; the CSV is only your data.


In [ ]:
import os
from pathlib import Path
import random

# Try to install transformers if not present
try:
    from transformers import GPT2TokenizerFast
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
    from transformers import GPT2TokenizerFast

# Initialize GPT-2 fast tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# Locate stories dataset file (commonly named stories.txt, or files in a directory)
stories_content = ""
possible_paths = [
    Path("tiny_stories_next_40k.csv"),
    Path("data/stories.txt"),
    Path("./"),
]

# Fallback to search directories for .txt files containing stories
found_file = None
for p in possible_paths:
    if p.is_file():
        found_file = p
        break
    elif p.is_dir():
        txt_files = list(p.glob("*.txt"))
        if txt_files:
            found_file = txt_files[0]
            break

if found_file and found_file.is_file():
    print(f"Reading stories from {found_file}")
    with open(found_file, "r", encoding="utf-8") as f:
        stories_content = f.read()
else:
    # Fallback template if no text file is found
    print("Stories dataset file not found in paths. Creating a sample story dataset...")
    sample_stories = [
        "Once upon a time, in a bright forest, there lived a tiny rabbit who loved to jump over blue rivers and sing songs to the giant trees.",
        "A brave knight set out on a grand adventure to find the golden crystal hidden deep within the mountain of whispers, protected by a friendly dragon.",
        "The clever little girl decided to build a solar powered rocket ship in her backyard using recycled metal, old toys, and cardboard boxes."
    ]
    stories_content = "\n\n".join(sample_stories * 50)

# Split into individual stories if separated by double-newlines
stories = [s.strip() for s in stories_content.split("\n\n") if s.strip()]

CONTEXT_WINDOW = 30
examples = []

print("Processing stories...")
stories_amount = 0
for story in stories:
    if stories_amount >= 100:
      break
    # Encode story into GPT-2 token IDs
    tokens = tokenizer.encode(story)
    if len(tokens) <= CONTEXT_WINDOW:
        continue

    # Slide the window across the story till the end
    for i in range(len(tokens) - CONTEXT_WINDOW):
        context_tokens = tokens[i : i + CONTEXT_WINDOW]
        next_token = tokens[i + CONTEXT_WINDOW]

        # Decode context to string representation for input, and target token to string representation
        # We can also keep them as token ID sequences if using BPE integration in BallTrapBrain
        input_str = tokenizer.decode(context_tokens)
        output_str = tokenizer.decode([next_token])

        examples.append((input_str, output_str))
    stories_amount += 1

# Shuffle prediction pairs
random.shuffle(examples)

# Update TOKENIZER variable so the subsequent cell can use it
TOKENIZER = tokenizer

if len(examples) < 3:
    raise ValueError("Provide at least three examples to create train, validation, and test splits.")
print(f"Loaded {len(examples):,} input/output next-token prediction pairs.")
print("First context input sample:", repr(examples[0][0]))
print("First target token sample:", repr(examples[0][1]))

Reading stories from tiny_stories_next_40k.csv
Processing stories...
Loaded 3,426 input/output next-token prediction pairs.
First context input sample: ' still got to have an adventure on a dark Tuesday!"\n"Once upon a time there was a friendly deer who lived in the forest. She liked'
First target token sample: ' to'


## 4. Choose the train/validation/test split

Set the fractions and seed here. The split is shuffled and performed before training. If examples from the same source must stay together (for example, multiple rows from one document), group them before applying this random row split.


In [ ]:
import random

SEED = 20261124
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

def split_pairs(pairs, validation_fraction=0.15, test_fraction=0.15, seed=0):
    if not (0 < validation_fraction < 1 and 0 <= test_fraction < 1
            and validation_fraction + test_fraction < 1):
        raise ValueError("Fractions must be nonnegative and sum to less than 1; validation must be positive.")
    order = list(range(len(pairs)))
    random.Random(seed).shuffle(order)
    n = len(order)
    n_val = max(1, round(n * validation_fraction))
    n_test = max(1, round(n * test_fraction)) if test_fraction else 0
    while n_val + n_test >= n:
        if n_test > 1:
            n_test -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            raise ValueError("Need at least three examples for nonempty train/validation/test sets.")
    val_ids = order[:n_val]
    test_ids = order[n_val:n_val+n_test]
    train_ids = order[n_val+n_test:]
    return ([pairs[i] for i in train_ids], [pairs[i] for i in val_ids],
            [pairs[i] for i in test_ids])

train_pairs, validation_pairs, test_pairs = split_pairs(
    examples, VALIDATION_FRACTION, TEST_FRACTION, SEED
)
print(f"Train: {len(train_pairs):,} | validation: {len(validation_pairs):,} | test: {len(test_pairs):,}")


Train: 2,398 | validation: 514 | test: 514


## 5. Configure and train

These are the main controls. `learning_rate`, `warmup_steps`, and `hard_steps` control optimization. The validation set selects the best rounded checkpoint; test data is held back for the final report.

The default is character-level text. The embedded brain also accepts a BPE tokenizer object: set `TOKENIZER` in the setup cell to an object exposing `encode` and `get_vocab` (such as `tokenizers.Tokenizer`). The optional tokenizer package/model is only needed if you choose BPE.

For larger text inputs, memory grows with the number of input tokens times the context-window width. Start with a smaller `D`, `hash_radius`, and step count, then scale up.


In [ ]:
# Architecture
GLOBAL_D = 50024
HASH_RADIUS = 5
DIRECTION_RANGE = 100
CAP = 50
BOUNCE = 10

# Optimization
LEARNING_RATE = 2
WARMUP_STEPS = 100
HARD_STEPS = 3600
CHECK_EVERY = 10

# Default: character-level text. For BPE, assign a tokenizer object to TOKENIZER
# (for example, a `tokenizers.Tokenizer`) before running this cell.

if TOKENIZER is None:
    input_alphabet = sorted(set("".join(prompt for prompt, _ in examples)))
    output_alphabet = sorted(set("".join(target for _, target in examples)))
    max_input_len = max(len(prompt) for prompt, _ in examples)
    if not input_alphabet or not output_alphabet:
        raise ValueError("Inputs and targets must contain at least one character overall.")
    brain = BallTrapBrain(
        input_alphabet, output_alphabet, D=GLOBAL_D, max_input_len=max_input_len,
        hash_radius=HASH_RADIUS, bounce=BOUNCE, cap=CAP,
        direction_range=DIRECTION_RANGE,
    )
else:
    max_input_len = max(len(encode_bpe(TOKENIZER, prompt)) for prompt, _ in examples)
    brain = BallTrapBrain.from_tokenizer(
        TOKENIZER, D=GLOBAL_D, max_input_len=max_input_len,
        hash_radius=HASH_RADIUS, bounce=BOUNCE, cap=CAP,
        direction_range=DIRECTION_RANGE,
    )

random.seed(SEED)
np.random.seed(SEED)
trainer = ContinuousBallTrainer(brain, train_pairs, auto_cap=True)
validation_matrix = CaseMatrix(trainer.brain, validation_pairs, encode_targets=False)
test_matrix = CaseMatrix(trainer.brain, test_pairs, encode_targets=False)

trainer.fit(
    warmup=WARMUP_STEPS,
    hard_steps=HARD_STEPS,
    learning_rate=LEARNING_RATE,
    validation=validation_matrix,
    check_every=CHECK_EVERY,
    verbose=True,
    batch_size=1
)
print(f"Effective cap after target-length check: {trainer.brain.cap}")
print(f"Training time: {trainer.seconds:.1f} seconds")


step=   0  hard train=44.204%  hard validation=3.502%  hard CE=0.0436
step= 100  hard train=44.204%  hard validation=3.502%  hard CE=0.0436


KeyboardInterrupt: 

## 6. Evaluate and compile to the hard brain

The relaxed trainer reports exact-match scores on train and validation data. This cell compiles the selected rounded router/table back into ordinary ball-trap deflectors, then measures train, validation, and test exact match on that discrete brain.


In [ ]:
def exact_accuracy(predictions, cases):
    return float(np.mean([pred == target for pred, (_, target) in zip(predictions, cases)])) if cases else float("nan")

hard_brain = trainer.compile()
train_predictions = [hard_brain.compute_input(x) for x, _ in train_pairs]
validation_predictions = [hard_brain.compute_input(x) for x, _ in validation_pairs]
test_predictions = [hard_brain.compute_input(x) for x, _ in test_pairs]

print(f"Compiled hard brain | D={hard_brain.D:,} | radius={hard_brain.hash_radius} | cap={hard_brain.cap}")
print(f"Train exact match:      {exact_accuracy(train_predictions, train_pairs):.2%}")
print(f"Validation exact match: {exact_accuracy(validation_predictions, validation_pairs):.2%}")
print(f"Test exact match:       {exact_accuracy(test_predictions, test_pairs):.2%}")

print("\nTest examples:")
for (prompt, expected), predicted in list(zip(test_pairs, test_predictions))[:20]:
    print(f"input={prompt!r} | expected={expected!r} | predicted={predicted!r}")


## 7. Try an input

After compilation, use `hard_brain.compute_input(...)` to run the native discrete ball-trap brain. For character mode, unseen input characters are ignored by the brain's input alphabet; include any desired input characters in the corpus alphabet or explicitly add them to `input_alphabet` before training.


In [ ]:
def generate(prompt):
    return hard_brain.compute_input(str(prompt))

# Change this prompt and rerun the cell.
PROMPT = test_pairs[0][0]
print("Input: ", PROMPT)
print("Output:", generate(PROMPT))
